<a href="https://colab.research.google.com/github/MeshMoh506/tiny-gpt/blob/main/tiny_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU memory: {total / 1e9:.1f} GB")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Sanity check: do a real computation on that device
x = torch.randn(1000, 1000, device=device)   # shape (1000, 1000)
y = x @ x                                    # matrix multiply
print("y shape :", y.shape, "| y device:", y.device)

Python : 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU    : Tesla T4
GPU memory: 15.6 GB
Using device: cuda
y shape : torch.Size([1000, 1000]) | y device: cuda:0


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/tiny-gpt"
for sub in ["checkpoints", "data", "outputs"]:
    os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Contents      :", sorted(os.listdir(PROJECT_DIR)))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/tiny-gpt
Contents      : ['checkpoints', 'data', 'outputs']


In [3]:
test_obj = {"note": "hello from stage 0", "step": 0, "tensor": torch.tensor([1.0, 2.0, 3.0])}
path = f"{PROJECT_DIR}/checkpoints/test.pt"

torch.save(test_obj, path)          # write to Drive
loaded = torch.load(path)           # read it back
print(loaded)
print("File size on Drive:", os.path.getsize(path), "bytes")

{'note': 'hello from stage 0', 'step': 0, 'tensor': tensor([1., 2., 3.])}
File size on Drive: 1620 bytes


In [4]:
import os, urllib.request

DATA_PATH = f"{PROJECT_DIR}/data/input.txt"
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Downloaded to Drive.")
else:
    print("Already on Drive, no download needed.")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print(text[:200])

Already on Drive, no download needed.
Total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [5]:
chars = sorted(set(text))
print(chars)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
chars = sorted(set(text))            # all unique characters, sorted
vocab_size = len(chars)
print("vocab_size:", vocab_size)
print(repr("".join(chars)))          # repr shows \n and spaces clearly

stoi = {ch: i for i, ch in enumerate(chars)}   # string -> integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer -> string

def encode(s):   return [stoi[c] for c in s]
def decode(ids): return "".join(itos[i] for i in ids)

ids = encode("hii there")
print(ids)
print(decode(ids))

# Sanity check: encode then decode must give back the same text
assert decode(encode(text[:1000])) == text[:1000]
print("Round-trip OK")

vocab_size: 65
"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there
Round-trip OK


In [7]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print("data shape:", data.shape, "| dtype:", data.dtype)
print(data[:20])
print("max id:", data.max().item(), "| min id:", data.min().item())

n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]
print("train:", len(train_data), "| val:", len(val_data))

data shape: torch.Size([1115394]) | dtype: torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])
max id: 64 | min id: 0
train: 1003854 | val: 111540


In [8]:
torch.manual_seed(1337)

block_size = 8     # T: how many characters of context
batch_size = 4     # B: how many chunks per batch

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))        # random start positions, shape (B,)
    x = torch.stack([d[i     : i + block_size    ] for i in ix])  # (B, T)
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])  # (B, T), shifted by 1
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x:", xb.shape, "| y:", yb.shape, "| device:", xb.device)
print(xb)
print(yb)

# Sanity check: y must equal x shifted by one position
assert torch.equal(xb[:, 1:], yb[:, :-1])
print("Shift check OK")

x: torch.Size([4, 8]) | y: torch.Size([4, 8]) | device: cuda:0
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]], device='cuda:0')
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]], device='cuda:0')
Shift check OK


In [9]:
for t in range(block_size):
    context = xb[0, : t + 1].tolist()
    target  = yb[0, t].item()
    print(f"context {decode(context)!r:>12}  ->  target {decode([target])!r}")

context          'L'  ->  target 'e'
context         'Le'  ->  target 't'
context        'Let'  ->  target "'"
context       "Let'"  ->  target 's'
context      "Let's"  ->  target ' '
context     "Let's "  ->  target 'h'
context    "Let's h"  ->  target 'e'
context   "Let's he"  ->  target 'a'


In [10]:
import math
import torch.nn as nn
import torch.nn.functional as F

class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)    # (65, 65) learnable table

    def forward(self, idx, targets=None):
        logits = self.table(idx)                             # (B, T) -> (B, T, C)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

torch.manual_seed(1337)
model = BigramLM(vocab_size).to(device)

xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("params :", sum(p.numel() for p in model.parameters()))
print("device :", next(model.parameters()).device)
print("logits :", logits.shape)
print("loss   :", loss.item())
print("ln(65) :", math.log(vocab_size))

params : 4225
device : cuda:0
logits : torch.Size([4, 8, 65])
loss   : 4.705134391784668
ln(65) : 4.174387269895637


In [11]:
batch_size = 32          # bigger batch = less noisy gradient (get_batch reads this global)

torch.manual_seed(1337)
model = BigramLM(vocab_size).to(device)          # fresh model, new random weights
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

losses = []
for step in range(3000):
    xb, yb = get_batch("train")                  # (32, 8)
    logits, loss = model(xb, yb)                 # forward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())                   # .item(): GPU scalar -> Python float
    if step % 500 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}")

print(f"avg of last 100 steps: {sum(losses[-100:]) / 100:.4f}")

step     0 | loss 4.6485
step   500 | loss 2.6240
step  1000 | loss 2.2548
step  1500 | loss 2.5394
step  2000 | loss 2.5437
step  2500 | loss 2.5022
avg of last 100 steps: 2.4675


In [12]:
with torch.no_grad():
    row = model.table.weight[stoi['t']]       # (65,) the logits row for 't'
    probs = F.softmax(row, dim=-1)            # (65,) sums to 1

top = torch.topk(probs, 5)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"after 't' -> {decode([i])!r}: {p:.2f}")

after 't' -> 'h': 0.36
after 't' -> ' ': 0.24
after 't' -> 'o': 0.08
after 't' -> 'e': 0.06
after 't' -> 'i': 0.04


In [13]:
@torch.no_grad()      # no gradients needed when generating: faster, less memory
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits, _ = model(idx)                              # (B, T, C)
        logits = logits[:, -1, :]                           # (B, C)  only the LAST position predicts the next char
        probs = F.softmax(logits, dim=-1)                   # (B, C)
        idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1) sample one char from the distribution
        idx = torch.cat([idx, idx_next], dim=1)             # (B, T+1) append and repeat
    return idx

torch.manual_seed(1337)
start = torch.zeros((1, 1), dtype=torch.long, device=device)   # a single token 0 = "\n"
out = generate(model, start, 300)
print("out shape:", out.shape)
print(decode(out[0].tolist()))

# Optional: save the trained weights to Drive
torch.save(model.state_dict(), f"{PROJECT_DIR}/checkpoints/bigram.pt")

out shape: torch.Size([1, 301])



CEThik brid owindakis s, ble

Hiset bube d e.
S:
O:3 my d?
LUCous:
Wanthar u qur, vet?
F dXENDoate awice my.

Hastarom orou wabuts, tof is h ble mil ndill, ath iree sengmin lat Heriliovets, and Win nghir.
Swanousel lind me l.
HAshe ce hiry:
Sugr aisspllwhy.
Hentous n Boopetelaves
MPOLI s, d mothak
